# 06 — Partición temporal del dataset

**Proyecto:** RepostaPro — Optimización del repostaje en flotas comerciales  
**Autor:** Víctor González Martín  
**Notebook:** 06 — Partición train / validation / test

## Objetivo del notebook

Particionar los datasets analíticos generados en la Fase 2 (uno por carburante) respetando la cronología temporal y la separación por régimen geopolítico. El resultado son 12 subconjuntos (2 carburantes × 2 régimenes × 3 particiones) que constituyen las entradas finales del modelado predictivo.

## Estrategia de partición

Para series temporales NO se aplica partición aleatoria. La estrategia adoptada respeta el orden cronológico:

**Régimen pre-shock** (787 días, 1 ene 2024 → 28 feb 2026):
- **Train** (75 %): 1 ene 2024 → 30 sep 2025
- **Validation** (12,5 %): 1 oct 2025 → 31 dic 2025
- **Test** (12,5 %): 1 ene 2026 → 28 feb 2026

**Régimen post-shock** (106 días, 1 mar 2026 → 14 jun 2026):
- **Train** (~58 %): 1 mar 2026 → 30 abr 2026
- **Validation** (~22 %): 1 may 2026 → 24 may 2026
- **Test** (~20 %): 25 may 2026 → 14 jun 2026

## Salida

Doce ficheros Parquet en `data/processed/particiones/`:
- `gasoleo_a_{pre_shock,post_shock}_{train,val,test}.parquet`
- `gasolina_95_{pre_shock,post_shock}_{train,val,test}.parquet`

In [1]:
# Configuración del entorno
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Añadir la raíz del proyecto al sys.path
RAIZ_PROYECTO = Path("..").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

# Importar funciones del módulo de partición
from src.particion import (particionar_pre_shock,particionar_post_shock,resumen_particion,guardar_particiones,)

# Rutas del proyecto
CARPETA_FEATURES = Path("../data/processed/features")
CARPETA_PARTICIONES = Path("../data/processed/particiones")

print(f"Raíz del proyecto: {RAIZ_PROYECTO}")
print(f"Carpeta de features:       {CARPETA_FEATURES.resolve()}")
print(f"Carpeta de particiones:    {CARPETA_PARTICIONES.resolve()}")
print("\n✓ Módulo src.particion importado correctamente")

Raíz del proyecto: C:\TFM
Carpeta de features:       C:\TFM\data\processed\features
Carpeta de particiones:    C:\TFM\data\processed\particiones

✓ Módulo src.particion importado correctamente


## Carga de los datasets de features

Cargamos los dos datasets analíticos generados en la Fase 2: uno por carburante del núcleo predictivo.

In [2]:
# Cargar los datasets de features
print("Cargando datasets de features...")

df_gasoleo = pd.read_parquet(CARPETA_FEATURES / "features_gasoleo_a.parquet")
df_gasolina = pd.read_parquet(CARPETA_FEATURES / "features_gasolina_95.parquet")

print(f"\nDatasets cargados")
print(f"  Gasóleo A:      {len(df_gasoleo):>10,} filas")
print(f"  Gasolina 95 E5: {len(df_gasolina):>10,} filas")

Cargando datasets de features...

✓ Datasets cargados
  Gasóleo A:       9,858,873 filas
  Gasolina 95 E5:  9,529,417 filas


## Partición del dataset de Gasóleo A

Aplicamos la partición temporal a los dos régimenes del Gasóleo A.

In [4]:
# Partición Gasóleo A
print(">>> Particionando Gasóleo A...\n")

# Pre-shock
particiones_pre_gasoleo = particionar_pre_shock(df_gasoleo)
resumen_particion(particiones_pre_gasoleo, "PRE-SHOCK · Gasóleo A")

# Post-shock
particiones_post_gasoleo = particionar_post_shock(df_gasoleo)
resumen_particion(particiones_post_gasoleo, "POST-SHOCK · Gasóleo A")

# Guardar
guardar_particiones(particiones_pre_gasoleo, "gasoleo_a", "pre_shock", CARPETA_PARTICIONES)
guardar_particiones(particiones_post_gasoleo, "gasoleo_a", "post_shock", CARPETA_PARTICIONES)
print("\nParticiones de Gasóleo A guardadas en data/processed/particiones/")

>>> Particionando Gasóleo A...


PARTICIÓN — Régimen PRE-SHOCK · Gasóleo A
  TRAIN         6,996,238 filas ( 80.7%)  2024-01-01 → 2025-09-30  (636 días)
  VAL           1,017,945 filas ( 11.7%)  2025-10-01 → 2025-12-31  (92 días)
  TEST            651,949 filas (  7.5%)  2026-01-01 → 2026-02-28  (59 días)

PARTICIÓN — Régimen POST-SHOCK · Gasóleo A
  TRAIN           687,425 filas ( 57.6%)  2026-03-01 → 2026-04-30  (61 días)
  VAL             269,279 filas ( 22.6%)  2026-05-01 → 2026-05-24  (24 días)
  TEST            236,037 filas ( 19.8%)  2026-05-25 → 2026-06-14  (21 días)

✓ Particiones de Gasóleo A guardadas en data/processed/particiones/


## Partición del dataset de Gasolina 95 E5

Aplicamos la partición temporal a los dos régimenes de la Gasolina 95 E5.

In [5]:
# Partición Gasolina 95 E5
print(">>> Particionando Gasolina 95 E5...\n")

# Pre-shock
particiones_pre_gasolina = particionar_pre_shock(df_gasolina)
resumen_particion(particiones_pre_gasolina, "PRE-SHOCK · Gasolina 95 E5")

# Post-shock
particiones_post_gasolina = particionar_post_shock(df_gasolina)
resumen_particion(particiones_post_gasolina, "POST-SHOCK · Gasolina 95 E5")

# Guardar
guardar_particiones(particiones_pre_gasolina, "gasolina_95", "pre_shock", CARPETA_PARTICIONES)
guardar_particiones(particiones_post_gasolina, "gasolina_95", "post_shock", CARPETA_PARTICIONES)
print("\nParticiones de Gasolina 95 E5 guardadas en data/processed/particiones/")

>>> Particionando Gasolina 95 E5...


PARTICIÓN — Régimen PRE-SHOCK · Gasolina 95 E5
  TRAIN         6,759,112 filas ( 80.7%)  2024-01-01 → 2025-09-30  (636 días)
  VAL             985,129 filas ( 11.8%)  2025-10-01 → 2025-12-31  (92 días)
  TEST            631,170 filas (  7.5%)  2026-01-01 → 2026-02-28  (59 días)

PARTICIÓN — Régimen POST-SHOCK · Gasolina 95 E5
  TRAIN           664,995 filas ( 57.6%)  2026-03-01 → 2026-04-30  (61 días)
  VAL             260,617 filas ( 22.6%)  2026-05-01 → 2026-05-24  (24 días)
  TEST            228,394 filas ( 19.8%)  2026-05-25 → 2026-06-14  (21 días)

✓ Particiones de Gasolina 95 E5 guardadas en data/processed/particiones/


## Verificación final: ficheros generados

Listamos los Parquet creados para confirmar que tenemos los 12 subconjuntos esperados.

In [2]:
# Verificación de los ficheros generados
ficheros_generados = sorted(CARPETA_PARTICIONES.glob("*.parquet"))

print(f"FICHEROS DE PARTICIONES GENERADOS")
print(f"=" * 70)
print(f"Total: {len(ficheros_generados)} ficheros\n")

print(f"{'Fichero':<55} {'Tamaño':>10}")
print("-" * 70)

for f in ficheros_generados:
    tamano_kb = f.stat().st_size / 1024
    if tamano_kb < 1024:
        tamano_str = f"{tamano_kb:.1f} KB"
    else:
        tamano_str = f"{tamano_kb/1024:.1f} MB"
    print(f"{f.name:<55} {tamano_str:>10}")

# Validación: ¿tenemos los 12 ficheros esperados?
esperados = 12
if len(ficheros_generados) == esperados:
    print(f"\n {esperados} ficheros generados correctamente.")
else:
    print(f"\nSe esperaban {esperados} ficheros, generados {len(ficheros_generados)}.")

FICHEROS DE PARTICIONES GENERADOS
Total: 12 ficheros

Fichero                                                     Tamaño
----------------------------------------------------------------------
gasoleo_a_post_shock_test.parquet                           3.6 MB
gasoleo_a_post_shock_train.parquet                          9.7 MB
gasoleo_a_post_shock_val.parquet                            3.8 MB
gasoleo_a_pre_shock_test.parquet                            7.2 MB
gasoleo_a_pre_shock_train.parquet                          84.9 MB
gasoleo_a_pre_shock_val.parquet                            10.6 MB
gasolina_95_post_shock_test.parquet                         3.4 MB
gasolina_95_post_shock_train.parquet                        8.9 MB
gasolina_95_post_shock_val.parquet                          3.6 MB
gasolina_95_pre_shock_test.parquet                          6.9 MB
gasolina_95_pre_shock_train.parquet                        79.2 MB
gasolina_95_pre_shock_val.parquet                           9.9 MB

 12